In [ ]:
import os

import random
import json
from typing import List, Dict, Any

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, LSTM, GlobalAveragePooling1D, Dense, Dropout, BatchNormalization, Concatenate, Layer, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [ ]:
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
tf.config.experimental.enable_op_determinism()

In [ ]:
# === Load and filter stock data ===
stocks = ['ENB','GS','WFC','GME','D','EA','CMCSA','DHI','CRM','VRTX',
          'SPWR','GILD','WDC','BX','AAL']

df = (
    pd.read_csv('fnspid_prices_title_sentiment.csv',
                parse_dates=['date'],
                index_col='date')
      .query("Stock_symbol in @stocks")
)

# === Generate binary target ===
def make_target(group):
    group['target_binary'] = (group['adj close'].shift(-1) > group['adj close']).astype(int)
    return group

df = (
    df.groupby('Stock_symbol', group_keys=False)
      .apply(make_target)
      .dropna(subset=['target_binary'])
)

# === Helper to load and daily-resample macro data ===
def load_macro(path, date_col='observation_date'):
    return (
        pd.read_csv(path, parse_dates=[date_col], index_col=date_col)
          .resample('D')
          .fillna(method='ffill').fillna(method='bfill')
    )

# === Load and combine all macro series ===
macro_paths = {
    'DFF': 'interest_rates.csv',
    'CPIAUCSL': 'cpi.csv',
    'UNRATE': 'unemployment_rate.csv',
    'PPI': 'ppi.csv',
    'GOLD_OIL_RATES': 'gold_silver_rates_oil_1999_2024.csv',
    'DTWEXBGS': 'nominal_us_dollar_index.csv'
}
macro_dfs = [load_macro(p) for p in macro_paths.values()]
merged_macro = (
    pd.concat(macro_dfs, axis=1)
      .reindex(df.index.unique())  # match only stock dates
      .fillna(method='ffill')      # forward-fill missing macro values
      .fillna(method='bfill')      # backfill start gaps if any
)

# === Join stock data with macro data ===
merged = df.join(merged_macro, how='left')
merged = merged.drop(columns=['Vol._x', 'Vol._y'])
# === Check ===
print(merged.info())
print(merged.head())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 33660 entries, 2015-01-05 to 2023-12-01
Data columns (total 39 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   volume                           33660 non-null  float64
 1   open                             33660 non-null  float64
 2   high                             33660 non-null  float64
 3   low                              33660 non-null  float64
 4   close                            33660 non-null  float64
 5   adj close                        33660 non-null  float64
 6   Stock_symbol                     33660 non-null  object 
 7   avg_weighted_sent                33660 non-null  float64
 8   avg_score                        33660 non-null  float64
 9   article_count                    33660 non-null  float64
 10  movement_percent                 33660 non-null  float64
 11  target_binary                    33660 non-null  int64  
 12  D

/tmp/ipython-input-634012813.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(make_target)
/tmp/ipython-input-634012813.py:28: FutureWarning: DatetimeIndexResampler.fillna is deprecated and will be removed in a future version. Use obj.ffill(), obj.bfill(), or obj.nearest() instead.
  .fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-634012813.py:28: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-634012813.py:28: FutureWarning: DatetimeIndexResampler.fillna is deprecated and will be removed in a future version. Use o

In [ ]:
# ======================
# Reproducibility
# ======================
def set_seed(seed: int):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

# ======================
# Sequence generator
# ======================
def make_sequences_dual(df, window_size, price_cols, sent_cols, target, skip_all_zero_sent=True):
    X_price, X_sent, y = [], [], []
    for i in range(len(df) - window_size):
        price_seq = df.iloc[i:i+window_size][price_cols].values
        sent_seq = df.iloc[i:i+window_size][sent_cols].values
        if skip_all_zero_sent and np.all(sent_seq == 0):
            continue
        target_val = df.iloc[i+window_size][target]
        X_price.append(price_seq)
        X_sent.append(sent_seq)
        y.append(target_val)
    return np.array(X_price), np.array(X_sent), np.array(y)

# ======================
# Temporal Attention Layer
# ======================
class TemporalAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], input_shape[-1]),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(shape=(input_shape[-1],),
                                 initializer='zeros', trainable=True)
        self.u = self.add_weight(shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform', trainable=True)
        super().build(input_shape)

    def call(self, x):
        uit = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        ait = tf.nn.softmax(tf.tensordot(uit, self.u, axes=1), axis=1)
        out = tf.reduce_sum(x * ait, axis=1)
        return out

# ======================
# Feature Attention Layer
# ======================
class FeatureAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], input_shape[-1]),
                                 initializer='glorot_uniform',
                                 trainable=True)
        self.b = self.add_weight(shape=(input_shape[-1],),
                                 initializer='zeros', trainable=True)
        self.u = self.add_weight(shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform',
                                 trainable=True)
        super().build(input_shape)

    def call(self, x):
        # x: (batch, timesteps, features)
        uit = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)  # (batch, timesteps, features)
        ait = tf.nn.softmax(tf.tensordot(uit, self.u, axes=1), axis=-1)  # attention over features
        out = x * ait
        return out

# ======================
# Model builder with feature attention
# ======================
def build_model_from_config(window_size, price_n_features, sent_n_features, config):
    l2_value = config.get('l2_value', 1e-4)
    learning_rate = config.get('learning_rate', 1e-4)

    # Price branch
    price_in = Input(shape=(window_size, price_n_features), name='price_input')
    conv_outputs = []  # collect conv outputs if concat_conv=True
    x = price_in

    concat_conv = config.get('concat_conv', False)

    for layer_cfg in config.get('price_stack', []):
        t = layer_cfg['type'].lower()
        if t == 'conv1d':
            conv = Conv1D(filters=layer_cfg.get('filters', 16),
                          kernel_size=layer_cfg.get('kernel_size', 3),
                          activation=layer_cfg.get('activation', 'relu'),
                          padding=layer_cfg.get('padding', 'same'))(x)
            conv = BatchNormalization()(conv) if layer_cfg.get('batchnorm', True) else conv
            if concat_conv:
                conv_outputs.append(conv)
            else:
                x = conv
        elif t == 'batchnorm' and not concat_conv:
            x = BatchNormalization()(x)
        elif t == 'dropout' and not concat_conv:
            x = Dropout(rate=layer_cfg.get('rate', 0.2))(x)
        elif t in ['lstm', 'gru']:
            break  # stop before recurrent layers

    # Concatenate conv outputs if requested
    if concat_conv and conv_outputs:
        x = Concatenate()(conv_outputs)

    # Feature attention
    if config.get('feature_attention', False):
        x = FeatureAttention()(x)

    # Apply recurrent layers
    rnn_layers = [l for l in config.get('price_stack', []) if l['type'].lower() in ['lstm', 'gru']]
    for i, layer_cfg in enumerate(rnn_layers):
        t = layer_cfg['type'].lower()
        return_seq = layer_cfg.get('return_sequences', True)
        if i == len(rnn_layers) - 1:
            return_seq = False
        units = layer_cfg.get('units', 32)
        rec_dropout = layer_cfg.get('recurrent_dropout', 0.0)
        if t == 'lstm':
            x = LSTM(units, recurrent_dropout=rec_dropout, return_sequences=return_seq)(x)
        elif t == 'gru':
            x = GRU(units, recurrent_dropout=rec_dropout, return_sequences=return_seq)(x)

    # Sentiment branch
    sent_in = Input(shape=(window_size, sent_n_features), name='sent_input')
    s = sent_in
    for layer_cfg in config.get('sent_stack', []):
        t = layer_cfg['type'].lower()
        if t == 'lstm':
            s = LSTM(layer_cfg.get('units', 8),
                     recurrent_dropout=layer_cfg.get('recurrent_dropout', 0.0),
                     return_sequences=layer_cfg.get('return_sequences', True))(s)
        elif t == 'dropout':
            s = Dropout(rate=layer_cfg.get('rate', 0.2))(s)
        elif t == 'batchnorm':
            s = BatchNormalization()(s)

    # Apply temporal attention to sent branch
    if config.get('sent_attention', True):
        s = TemporalAttention()(s)
    else:
        s = LSTM(config.get('sent_final_lstm_units', 8), return_sequences=False)(s)

    # Merge and output
    merged = Concatenate()([x, s])
    out = Dense(1, activation='sigmoid', kernel_regularizer=l2(l2_value))(merged)

    model = Model(inputs=[price_in, sent_in], outputs=out)
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['AUC', 'accuracy'])
    return model



In [ ]:
def overlapping_fixed_tscv(n_samples, n_splits, train_size, val_size, test_size):
    """
    Fixed-size overlapping rolling time series splits.
    Each fold shifts forward by step = test_size.
    """
    total_window = train_size + val_size + test_size
    step = test_size  # overlap controlled by test window

    for i in range(n_splits):
        start = i * step
        end = start + total_window
        if end > n_samples:
            break
        train_idx = np.arange(start, start + train_size)
        val_idx = np.arange(start + train_size, start + train_size + val_size)
        test_idx = np.arange(start + train_size + val_size, start + total_window)
        yield train_idx, val_idx, test_idx

# ======================
# Training loop (fixed)
# ======================
def train_on_merged(merged, config):
    set_seed(config.get('seed', 0))
    window_size = config['window_size']
    price_cols = config['price_cols']
    macro_cols = config['macro_cols']
    sent_cols = config['sent_cols']
    target = config['target']
    feature_cols = config['feature_cols'] or [f"{c}_logret" for c in price_cols] + [f"{c}_ret" for c in macro_cols]

    results = []

    for symbol in merged['Stock_symbol'].unique():
        print(f"\n==== Training for: {symbol} ====")
        df_symbol = merged[merged['Stock_symbol'] == symbol].copy()

        # Compute returns
        for col in price_cols:
            safe = df_symbol[col].replace(0, np.nan)
            df_symbol[f'{col}_logret'] = np.log(safe) - np.log(safe.shift(1))
        for col in macro_cols:
            df_symbol[f'{col}_ret'] = df_symbol[col].pct_change()
        df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')

        Xp_full, Xs_full, y_full = make_sequences_dual(df_symbol, window_size, feature_cols, sent_cols, target)
        if len(Xp_full) == 0:
            print(f"No sequences for {symbol}, skipping.")
            continue

        X_idx = np.arange(len(Xp_full))

        fold = 0
        for train_idx, val_idx, test_idx in overlapping_fixed_tscv(
                n_samples=len(X_idx),
                n_splits=5,
                train_size=1000,
                val_size=150,
                test_size=150):

            fold += 1
            print(f"\n-- Fold {fold} --")
            Xp_train, Xp_val, Xp_test = Xp_full[train_idx], Xp_full[val_idx], Xp_full[test_idx]
            Xs_train, Xs_val, Xs_test = Xs_full[train_idx], Xs_full[val_idx], Xs_full[test_idx]
            y_train, y_val, y_test = y_full[train_idx], y_full[val_idx], y_full[test_idx]

            def dist(name, y):
                ones = np.sum(y == 1)
                zeros = np.sum(y == 0)
                print(f"{name} -> 1: {ones/len(y)*100:.2f}% | 0: {zeros/len(y)*100:.2f}% (n={len(y)})")

            dist("Train", y_train)
            dist("Validation", y_val)
            dist("Test", y_test)

            model = build_model_from_config(window_size, len(feature_cols), len(sent_cols), config)
            early_stop = EarlyStopping(
                monitor='val_loss',
                patience=config['early_stopping_patience'],
                restore_best_weights=True
            )

            model.fit(
                [Xp_train, Xs_train], y_train,
                validation_data=([Xp_val, Xs_val], y_val),
                epochs=config['epochs'],
                batch_size=config['batch_size'],
                callbacks=[early_stop],
                verbose=0
            )

            y_prob = model.predict([Xp_test, Xs_test])
            y_pred = (y_prob > 0.5).astype(int)

            auc = roc_auc_score(y_test, y_prob)
            f1 = f1_score(y_test, y_pred)
            mcc = matthews_corrcoef(y_test, y_pred)
            acc = np.mean(y_pred.flatten() == y_test.flatten())
            cm = confusion_matrix(y_test, y_pred)

            print(f"[{symbol}][Fold {fold}] AUC: {auc:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | ACC: {acc:.4f}")
            print(f"Confusion matrix:\n{cm}\n")

            results.append({
                'Symbol': symbol,
                'Fold': fold,
                'Test_AUC': auc,
                'Test_F1': f1,
                'Test_MCC': mcc,
                'Test_ACC': acc
            })

    results_df = pd.DataFrame(results)
    return results_df


In [ ]:
def summarize_results(results_df):
    """Summarize per-stock and overall mean AUC and ACC."""
    grouped = results_df.groupby('Symbol')

    acc_by_symbol = grouped['Test_ACC'].mean().sort_values(ascending=False)
    auc_by_symbol = grouped['Test_AUC'].mean().sort_values(ascending=False)
    mcc_by_symbol = grouped['Test_MCC'].mean().sort_values(ascending=False)
    f1_by_symbol = grouped['Test_F1'].mean().sort_values(ascending=False)


    print("=== Per-Stock Summary ===")
    print("\nMean Test Accuracy by Symbol:")
    print(acc_by_symbol)

    print("\nMean Test AUC by Symbol:")
    print(auc_by_symbol)

    print("\nMean Test MCC by Symbol:")
    print(mcc_by_symbol)

    print("\nMean Test F1 by Symbol:")
    print(f1_by_symbol)

    print("\n=== Overall Summary ===")
    print(f"Overall Mean ACC: {acc_by_symbol.mean():.4f}")
    print(f"Overall Mean AUC: {auc_by_symbol.mean():.4f}")
    print(f"Overall Mean MCC: {mcc_by_symbol.mean():.4f}")
    print(f"Overall Mean F1: {f1_by_symbol.mean():.4f}")

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 5,
    "price_cols": ['open', 'high', 'low', 'close', 'volume'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}

# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 42.67% | 0: 57.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 177ms/step
[AAL][Fold 1] AUC: 0.4893 | F1: 0.4065 | MCC: -0.0048 | ACC: 0.5133
Confusion matrix:
[[52 34]
 [39 25]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 42.67% | 0: 57.33% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 175ms/step
[AAL][Fold 2] AUC: 0.5311 | F1: 0.2105 | MCC: 0.0891 | ACC: 0.5000
Confusion matrix:
[[65  5]
 [70 10]]


-- Fold 3 --
Train -> 1: 47.90% | 0: 52.10% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 44.00% | 0: 56.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 183ms/step
[AAL][Fold 3] AUC: 0.4939 | F1: 0.2174 | MCC: -0.0511 | ACC: 0.5200
Confusion matrix:
[[68 16]
 [56 10]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 44.00% | 0: 56.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 57.33% | 0: 42.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 167ms/step
[BX][Fold 1] AUC: 0.5356 | F1: 0.7109 | MCC: 0.1206 | ACC: 0.5933
Confusion matrix:
[[14 50]
 [11 75]]


-- Fold 2 --
Train -> 1: 52.40% | 0: 47.60% (n=1000)
Validation -> 1: 57.33% | 0: 42.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 169ms/step
[BX][Fold 2] AUC: 0.5754 | F1: 0.6393 | MCC: 0.0766 | ACC: 0.4733
Confusion matrix:
[[ 1 79]
 [ 0 70]]


-- Fold 3 --
Train -> 1: 53.80% | 0: 46.20% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 166ms/step
[BX][Fold 3] AUC: 0.5087 | F1: 0.5591 | MCC: -0.0850 | ACC: 0.4533
Confusion matrix:
[[16 62]
 [20 52]]


-- Fold 4 --
Train -> 1: 54.10% | 0: 45.90% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1

...


-- Fold 1 --
Train -> 1: 50.90% | 0: 49.10% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 183ms/step
[VRTX][Fold 1] AUC: 0.5072 | F1: 0.5902 | MCC: 0.0141 | ACC: 0.5000
Confusion matrix:
[[21 56]
 [19 54]]


-- Fold 2 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 180ms/step
[VRTX][Fold 2] AUC: 0.5306 | F1: 0.6484 | MCC: 0.1102 | ACC: 0.4867
Confusion matrix:
[[ 2 77]
 [ 0 71]]


-- Fold 3 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 176ms/step
[VRTX][Fold 3] AUC: 0.5616 | F1: 0.7273 | MCC: -0.0153 | ACC: 0.5800
Confusion matrix:
[[ 3 58]
 [ 5 84]]


-- Fold 4 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 59.33% | 0: 40.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 175ms/step
[WDC][Fold 1] AUC: 0.4852 | F1: 0.5600 | MCC: -0.0327 | ACC: 0.4867
Confusion matrix:
[[24 50]
 [27 49]]


-- Fold 2 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 170ms/step
[WDC][Fold 2] AUC: 0.4466 | F1: 0.5856 | MCC: 0.0204 | ACC: 0.5000
Confusion matrix:
[[22 56]
 [19 53]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 244ms/step
[WDC][Fold 3] AUC: 0.4378 | F1: 0.5581 | MCC: -0.0097 | ACC: 0.4933
Confusion matrix:
[[26 50]
 [26 48]]


-- Fold 4 --
Train -> 1: 51.00% | 0: 49.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 50.00% | 0: 50.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 43.33% | 0: 56.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 167ms/step
[WFC][Fold 1] AUC: 0.4552 | F1: 0.4276 | MCC: -0.0989 | ACC: 0.4467
Confusion matrix:
[[36 49]
 [34 31]]


-- Fold 2 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 43.33% | 0: 56.67% (n=150)
Test -> 1: 52.67% | 0: 47.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 170ms/step
[WFC][Fold 2] AUC: 0.5315 | F1: 0.1702 | MCC: 0.0045 | ACC: 0.4800
Confusion matrix:
[[64  7]
 [71  8]]


-- Fold 3 --
Train -> 1: 48.80% | 0: 51.20% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 171ms/step
[WFC][Fold 3] AUC: 0.4673 | F1: 0.4780 | MCC: -0.1103 | ACC: 0.4467
Confusion matrix:
[[29 40]
 [43 38]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 5,
    "price_cols": ['open', 'high', 'low', 'close', 'volume'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 42.67% | 0: 57.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 235ms/step
[AAL][Fold 1] AUC: 0.5594 | F1: 0.4478 | MCC: 0.0036 | ACC: 0.5067
Confusion matrix:
[[46 40]
 [34 30]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 42.67% | 0: 57.33% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 332ms/step
[AAL][Fold 2] AUC: 0.5350 | F1: 0.2626 | MCC: 0.1152 | ACC: 0.5133
Confusion matrix:
[[64  6]
 [67 13]]


-- Fold 3 --
Train -> 1: 47.90% | 0: 52.10% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 44.00% | 0: 56.00% (n=150)


1/5 ━━━━━━━━━━━━━━━━━━━━ 3s 859ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 229ms/step
[AAL][Fold 3] AUC: 0.4755 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5600
Confusion matrix:
[[84  0]
 [66  0]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 44.00% | 0: 56.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 325ms/step
[AAL][Fold 4] AUC: 0.4595 | F1: 0.1205 | MCC: -0.0804 | ACC: 0.5133
Confusion matrix:
[[72 10]
 [63  5]]


-- Fold 5 --
Train -> 1: 47.30% | 0: 52.70% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 235ms/step
[AAL][Fold 5] AUC: 0.4980 | F1: 0.0741 | MCC: -0.0287 | ACC: 0.5000
Confusion matrix:
[[72  4]
 [71  3]]


==== Training for: BX ====


...


-- Fold 1 --
Train -> 1: 50.90% | 0: 49.10% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 228ms/step
[VRTX][Fold 1] AUC: 0.5127 | F1: 0.5730 | MCC: -0.0029 | ACC: 0.4933
Confusion matrix:
[[23 54]
 [22 51]]


-- Fold 2 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 338ms/step
[VRTX][Fold 2] AUC: 0.5174 | F1: 0.6455 | MCC: 0.0777 | ACC: 0.4800
Confusion matrix:
[[ 1 78]
 [ 0 71]]


-- Fold 3 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 228ms/step
[VRTX][Fold 3] AUC: 0.4561 | F1: 0.7342 | MCC: -0.0962 | ACC: 0.5800
Confusion matrix:
[[ 0 61]
 [ 2 87]]


-- Fold 4 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 59.33% | 0: 40.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━

/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 228ms/step
[WDC][Fold 1] AUC: 0.4541 | F1: 0.5380 | MCC: -0.0590 | ACC: 0.4733
Confusion matrix:
[[25 49]
 [30 46]]


-- Fold 2 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 261ms/step
[WDC][Fold 2] AUC: 0.4995 | F1: 0.6516 | MCC: 0.0787 | ACC: 0.4867
Confusion matrix:
[[ 1 77]
 [ 0 72]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 244ms/step
[WDC][Fold 3] AUC: 0.5121 | F1: 0.6455 | MCC: -0.0850 | ACC: 0.4800
Confusion matrix:
[[ 1 75]
 [ 3 71]]


-- Fold 4 --
Train -> 1: 51.00% | 0: 49.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 50.00% | 0: 50.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 43.33% | 0: 56.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 234ms/step
[WFC][Fold 1] AUC: 0.4914 | F1: 0.5311 | MCC: -0.0474 | ACC: 0.4467
Confusion matrix:
[[20 65]
 [18 47]]


-- Fold 2 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 43.33% | 0: 56.67% (n=150)
Test -> 1: 52.67% | 0: 47.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 303ms/step
[WFC][Fold 2] AUC: 0.4628 | F1: 0.3415 | MCC: -0.0637 | ACC: 0.4600
Confusion matrix:
[[48 23]
 [58 21]]


-- Fold 3 --
Train -> 1: 48.80% | 0: 51.20% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 229ms/step
[WFC][Fold 3] AUC: 0.4824 | F1: 0.4552 | MCC: -0.0422 | ACC: 0.4733
Confusion matrix:
[[38 31]
 [48 33]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 5,
    "price_cols": ['open', 'high', 'low', 'close', 'volume'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}

# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 42.67% | 0: 57.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 331ms/step
[AAL][Fold 1] AUC: 0.5116 | F1: 0.1351 | MCC: 0.0396 | ACC: 0.5733
Confusion matrix:
[[81  5]
 [59  5]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 42.67% | 0: 57.33% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 406ms/step
[AAL][Fold 2] AUC: 0.4709 | F1: 0.0920 | MCC: 0.0169 | ACC: 0.4733
Confusion matrix:
[[67  3]
 [76  4]]


-- Fold 3 --
Train -> 1: 47.90% | 0: 52.10% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 44.00% | 0: 56.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 330ms/step
[AAL][Fold 3] AUC: 0.4876 | F1: 0.2653 | MCC: -0.0354 | ACC: 0.5200
Confusion matrix:
[[65 19]
 [53 13]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 44.00% | 0: 56.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 57.33% | 0: 42.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 293ms/step
[BX][Fold 1] AUC: 0.5191 | F1: 0.6700 | MCC: 0.0518 | ACC: 0.5600
Confusion matrix:
[[17 47]
 [19 67]]


-- Fold 2 --
Train -> 1: 52.40% | 0: 47.60% (n=1000)
Validation -> 1: 57.33% | 0: 42.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 388ms/step
[BX][Fold 2] AUC: 0.6050 | F1: 0.6117 | MCC: -0.0214 | ACC: 0.4667
Confusion matrix:
[[ 7 73]
 [ 7 63]]


-- Fold 3 --
Train -> 1: 53.80% | 0: 46.20% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 363ms/step
[BX][Fold 3] AUC: 0.6070 | F1: 0.6264 | MCC: 0.1267 | ACC: 0.5467
Confusion matrix:
[[25 53]
 [15 57]]


-- Fold 4 --
Train -> 1: 54.10% | 0: 45.90% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3

...


-- Fold 1 --
Train -> 1: 50.90% | 0: 49.10% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 321ms/step
[VRTX][Fold 1] AUC: 0.5467 | F1: 0.6010 | MCC: -0.0133 | ACC: 0.4867
Confusion matrix:
[[15 62]
 [15 58]]


-- Fold 2 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 324ms/step
[VRTX][Fold 2] AUC: 0.4928 | F1: 0.6204 | MCC: -0.1215 | ACC: 0.4533
Confusion matrix:
[[ 1 78]
 [ 4 67]]


-- Fold 3 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 313ms/step
[VRTX][Fold 3] AUC: 0.5868 | F1: 0.7401 | MCC: 0.1061 | ACC: 0.6067
Confusion matrix:
[[ 7 54]
 [ 5 84]]


-- Fold 4 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 59.33% | 0: 40.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━

/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 335ms/step
[WDC][Fold 1] AUC: 0.4390 | F1: 0.5497 | MCC: -0.0314 | ACC: 0.4867
Confusion matrix:
[[26 48]
 [29 47]]


-- Fold 2 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 372ms/step
[WDC][Fold 2] AUC: 0.5274 | F1: 0.6575 | MCC: 0.1373 | ACC: 0.5000
Confusion matrix:
[[ 3 75]
 [ 0 72]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 300ms/step
[WDC][Fold 3] AUC: 0.4291 | F1: 0.6606 | MCC: 0.0457 | ACC: 0.5000
Confusion matrix:
[[ 2 74]
 [ 1 73]]


-- Fold 4 --
Train -> 1: 51.00% | 0: 49.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 50.00% | 0: 50.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 43.33% | 0: 56.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 295ms/step
[WFC][Fold 1] AUC: 0.5714 | F1: 0.4074 | MCC: 0.1002 | ACC: 0.5733
Confusion matrix:
[[64 21]
 [43 22]]


-- Fold 2 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 43.33% | 0: 56.67% (n=150)
Test -> 1: 52.67% | 0: 47.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 291ms/step
[WFC][Fold 2] AUC: 0.4960 | F1: 0.2292 | MCC: 0.0862 | ACC: 0.5067
Confusion matrix:
[[65  6]
 [68 11]]


-- Fold 3 --
Train -> 1: 48.80% | 0: 51.20% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 293ms/step
[WFC][Fold 3] AUC: 0.5135 | F1: 0.5749 | MCC: 0.0422 | ACC: 0.5267
Confusion matrix:
[[31 38]
 [33 48]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 5,
    "price_cols": ['open', 'high', 'low', 'close', 'volume', 'DTWEXBGS', 'Price_x', 'WTI Crude Oil Price/Barrel'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)


==== Training for: AAL ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 42.67% | 0: 57.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 163ms/step
[AAL][Fold 1] AUC: 0.4729 | F1: 0.3761 | MCC: -0.0173 | ACC: 0.5133
Confusion matrix:
[[55 31]
 [42 22]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 42.67% | 0: 57.33% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 166ms/step
[AAL][Fold 2] AUC: 0.5079 | F1: 0.1856 | MCC: -0.0028 | ACC: 0.4733
Confusion matrix:
[[62  8]
 [71  9]]


-- Fold 3 --
Train -> 1: 47.90% | 0: 52.10% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 44.00% | 0: 56.00% (n=150)


1/5 ━━━━━━━━━━━━━━━━━━━━ 2s 624ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 170ms/step
[AAL][Fold 3] AUC: 0.5323 | F1: 0.4107 | MCC: 0.0804 | ACC: 0.5600
Confusion matrix:
[[61 23]
 [43 23]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 44.00% | 0: 56.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 164ms/step
[AAL][Fold 4] AUC: 0.5072 | F1: 0.1282 | MCC: 0.0251 | ACC: 0.5467
Confusion matrix:
[[77  5]
 [63  5]]


-- Fold 5 --
Train -> 1: 47.30% | 0: 52.70% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 168ms/step
[AAL][Fold 5] AUC: 0.5085 | F1: 0.0976 | MCC: 0.0032 | ACC: 0.5067
Confusion matrix:
[[72  4]
 [70  4]]


==== Training for: BX ====


...


-- Fold 1 --
Train -> 1: 50.90% | 0: 49.10% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 160ms/step
[VRTX][Fold 1] AUC: 0.4812 | F1: 0.5778 | MCC: -0.0022 | ACC: 0.4933
Confusion matrix:
[[22 55]
 [21 52]]


-- Fold 2 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 242ms/step
[VRTX][Fold 2] AUC: 0.5363 | F1: 0.6455 | MCC: 0.0777 | ACC: 0.4800
Confusion matrix:
[[ 1 78]
 [ 0 71]]


-- Fold 3 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 165ms/step
[VRTX][Fold 3] AUC: 0.5426 | F1: 0.7273 | MCC: -0.0153 | ACC: 0.5800
Confusion matrix:
[[ 3 58]
 [ 5 84]]


-- Fold 4 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 59.33% | 0: 40.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 187ms/step
[WDC][Fold 1] AUC: 0.4550 | F1: 0.6032 | MCC: -0.0078 | ACC: 0.5000
Confusion matrix:
[[18 56]
 [19 57]]


-- Fold 2 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 169ms/step
[WDC][Fold 2] AUC: 0.4348 | F1: 0.6269 | MCC: 0.0415 | ACC: 0.5000
Confusion matrix:
[[12 66]
 [ 9 63]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 165ms/step
[WDC][Fold 3] AUC: 0.4555 | F1: 0.6509 | MCC: 0.0452 | ACC: 0.5067
Confusion matrix:
[[ 7 69]
 [ 5 69]]


-- Fold 4 --
Train -> 1: 51.00% | 0: 49.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 50.00% | 0: 50.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 43.33% | 0: 56.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 212ms/step
[WFC][Fold 1] AUC: 0.4910 | F1: 0.4970 | MCC: -0.0895 | ACC: 0.4333
Confusion matrix:
[[23 62]
 [23 42]]


-- Fold 2 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 43.33% | 0: 56.67% (n=150)
Test -> 1: 52.67% | 0: 47.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 164ms/step
[WFC][Fold 2] AUC: 0.5245 | F1: 0.1522 | MCC: 0.0073 | ACC: 0.4800
Confusion matrix:
[[65  6]
 [72  7]]


-- Fold 3 --
Train -> 1: 48.80% | 0: 51.20% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 169ms/step
[WFC][Fold 3] AUC: 0.4609 | F1: 0.6051 | MCC: -0.0802 | ACC: 0.4867
Confusion matrix:
[[14 55]
 [22 59]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 5,
    "price_cols": ['open', 'high', 'low', 'close', 'volume', 'DTWEXBGS', 'Price_x', 'WTI Crude Oil Price/Barrel'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 42.67% | 0: 57.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 265ms/step
[AAL][Fold 1] AUC: 0.5374 | F1: 0.5306 | MCC: 0.0972 | ACC: 0.5400
Confusion matrix:
[[42 44]
 [25 39]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 42.67% | 0: 57.33% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 242ms/step
[AAL][Fold 2] AUC: 0.5400 | F1: 0.1364 | MCC: 0.1031 | ACC: 0.4933
Confusion matrix:
[[68  2]
 [74  6]]


-- Fold 3 --
Train -> 1: 47.90% | 0: 52.10% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 44.00% | 0: 56.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 238ms/step
[AAL][Fold 3] AUC: 0.4758 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5600
Confusion matrix:
[[84  0]
 [66  0]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 44.00% | 0: 56.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 57.33% | 0: 42.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 228ms/step
[BX][Fold 1] AUC: 0.5332 | F1: 0.7156 | MCC: 0.0962 | ACC: 0.5867
Confusion matrix:
[[10 54]
 [ 8 78]]


-- Fold 2 --
Train -> 1: 52.40% | 0: 47.60% (n=1000)
Validation -> 1: 57.33% | 0: 42.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 227ms/step
[BX][Fold 2] AUC: 0.5979 | F1: 0.6132 | MCC: -0.0753 | ACC: 0.4533
Confusion matrix:
[[ 3 77]
 [ 5 65]]


-- Fold 3 --
Train -> 1: 53.80% | 0: 46.20% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 228ms/step
[BX][Fold 3] AUC: 0.5591 | F1: 0.5955 | MCC: 0.0621 | ACC: 0.5200
Confusion matrix:
[[25 53]
 [19 53]]


-- Fold 4 --
Train -> 1: 54.10% | 0: 45.90% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2

...


-- Fold 1 --
Train -> 1: 50.90% | 0: 49.10% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 231ms/step
[VRTX][Fold 1] AUC: 0.5028 | F1: 0.6054 | MCC: 0.0458 | ACC: 0.5133
Confusion matrix:
[[21 56]
 [17 56]]


-- Fold 2 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 226ms/step
[VRTX][Fold 2] AUC: 0.4803 | F1: 0.6425 | MCC: 0.0000 | ACC: 0.4733
Confusion matrix:
[[ 0 79]
 [ 0 71]]


-- Fold 3 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 233ms/step
[VRTX][Fold 3] AUC: 0.4312 | F1: 0.6937 | MCC: -0.0819 | ACC: 0.5467
Confusion matrix:
[[ 5 56]
 [12 77]]


-- Fold 4 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 59.33% | 0: 40.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 232ms/step
[WDC][Fold 1] AUC: 0.4712 | F1: 0.5393 | MCC: -0.1052 | ACC: 0.4533
Confusion matrix:
[[20 54]
 [28 48]]


-- Fold 2 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 231ms/step
[WDC][Fold 2] AUC: 0.4897 | F1: 0.6275 | MCC: 0.0263 | ACC: 0.4933
Confusion matrix:
[[10 68]
 [ 8 64]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 229ms/step
[WDC][Fold 3] AUC: 0.5048 | F1: 0.6636 | MCC: 0.1034 | ACC: 0.5200
Confusion matrix:
[[ 7 69]
 [ 3 71]]


-- Fold 4 --
Train -> 1: 51.00% | 0: 49.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 50.00% | 0: 50.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 43.33% | 0: 56.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 11s 270ms/step
[WFC][Fold 1] AUC: 0.4977 | F1: 0.5405 | MCC: -0.0673 | ACC: 0.4333
Confusion matrix:
[[15 70]
 [15 50]]


-- Fold 2 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 43.33% | 0: 56.67% (n=150)
Test -> 1: 52.67% | 0: 47.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 257ms/step
[WFC][Fold 2] AUC: 0.4776 | F1: 0.2353 | MCC: -0.0042 | ACC: 0.4800
Confusion matrix:
[[60 11]
 [67 12]]


-- Fold 3 --
Train -> 1: 48.80% | 0: 51.20% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 248ms/step
[WFC][Fold 3] AUC: 0.4806 | F1: 0.6283 | MCC: 0.0181 | ACC: 0.5267
Confusion matrix:
[[19 50]
 [21 60]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 5,
    "price_cols": ['open', 'high', 'low', 'close', 'volume', 'DTWEXBGS', 'Price_x', 'WTI Crude Oil Price/Barrel'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 42.67% | 0: 57.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 289ms/step
[AAL][Fold 1] AUC: 0.5213 | F1: 0.2529 | MCC: 0.0444 | ACC: 0.5667
Confusion matrix:
[[74 12]
 [53 11]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 42.67% | 0: 57.33% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 300ms/step
[AAL][Fold 2] AUC: 0.4643 | F1: 0.1149 | MCC: 0.0802 | ACC: 0.4867
Confusion matrix:
[[68  2]
 [75  5]]


-- Fold 3 --
Train -> 1: 47.90% | 0: 52.10% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 44.00% | 0: 56.00% (n=150)


1/5 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 301ms/step
[AAL][Fold 3] AUC: 0.5081 | F1: 0.1299 | MCC: 0.0082 | ACC: 0.5533
Confusion matrix:
[[78  6]
 [61  5]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 44.00% | 0: 56.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 290ms/step
[AAL][Fold 4] AUC: 0.4962 | F1: 0.0556 | MCC: 0.0155 | ACC: 0.5467
Confusion matrix:
[[80  2]
 [66  2]]


-- Fold 5 --
Train -> 1: 47.30% | 0: 52.70% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 961ms/step
[AAL][Fold 5] AUC: 0.4611 | F1: 0.2000 | MCC: 0.0478 | ACC: 0.5200
Confusion matrix:
[[69  7]
 [65  9]]


==== Training for: BX ====


...


-- Fold 1 --
Train -> 1: 50.90% | 0: 49.10% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 298ms/step
[VRTX][Fold 1] AUC: 0.5238 | F1: 0.6321 | MCC: 0.0867 | ACC: 0.5267
Confusion matrix:
[[18 59]
 [12 61]]


-- Fold 2 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 424ms/step
[VRTX][Fold 2] AUC: 0.4974 | F1: 0.6359 | MCC: -0.0088 | ACC: 0.4733
Confusion matrix:
[[ 2 77]
 [ 2 69]]


-- Fold 3 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 311ms/step
[VRTX][Fold 3] AUC: 0.5832 | F1: 0.7401 | MCC: 0.1061 | ACC: 0.6067
Confusion matrix:
[[ 7 54]
 [ 5 84]]


-- Fold 4 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 59.33% | 0: 40.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 296ms/step
[WDC][Fold 1] AUC: 0.4467 | F1: 0.5122 | MCC: -0.0700 | ACC: 0.4667
Confusion matrix:
[[28 46]
 [34 42]]


-- Fold 2 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 298ms/step
[WDC][Fold 2] AUC: 0.5198 | F1: 0.6573 | MCC: 0.1304 | ACC: 0.5133
Confusion matrix:
[[ 7 71]
 [ 2 70]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 296ms/step
[WDC][Fold 3] AUC: 0.4168 | F1: 0.6606 | MCC: 0.0457 | ACC: 0.5000
Confusion matrix:
[[ 2 74]
 [ 1 73]]


-- Fold 4 --
Train -> 1: 51.00% | 0: 49.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 50.00% | 0: 50.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 43.33% | 0: 56.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 292ms/step
[WFC][Fold 1] AUC: 0.5388 | F1: 0.3434 | MCC: 0.0728 | ACC: 0.5667
Confusion matrix:
[[68 17]
 [48 17]]


-- Fold 2 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 43.33% | 0: 56.67% (n=150)
Test -> 1: 52.67% | 0: 47.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 298ms/step
[WFC][Fold 2] AUC: 0.5076 | F1: 0.1758 | MCC: 0.0827 | ACC: 0.5000
Confusion matrix:
[[67  4]
 [71  8]]


-- Fold 3 --
Train -> 1: 48.80% | 0: 51.20% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 367ms/step
[WFC][Fold 3] AUC: 0.4178 | F1: 0.5464 | MCC: -0.1457 | ACC: 0.4467
Confusion matrix:
[[17 52]
 [31 50]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 5,
    "price_cols": ['open', 'high', 'low', 'close', 'volume', 'DTWEXBGS', 'Price_x', 'WTI Crude Oil Price/Barrel'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.0, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.0, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.0, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 42.67% | 0: 57.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 159ms/step
[AAL][Fold 1] AUC: 0.5371 | F1: 0.4593 | MCC: 0.0191 | ACC: 0.5133
Confusion matrix:
[[46 40]
 [33 31]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 42.67% | 0: 57.33% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 200ms/step
[AAL][Fold 2] AUC: 0.5514 | F1: 0.2653 | MCC: 0.1398 | ACC: 0.5200
Confusion matrix:
[[65  5]
 [67 13]]


-- Fold 3 --
Train -> 1: 47.90% | 0: 52.10% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 44.00% | 0: 56.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 141ms/step
[AAL][Fold 3] AUC: 0.4742 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5600
Confusion matrix:
[[84  0]
 [66  0]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 44.00% | 0: 56.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 57.33% | 0: 42.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step
[BX][Fold 1] AUC: 0.5461 | F1: 0.7042 | MCC: 0.0818 | ACC: 0.5800
Confusion matrix:
[[12 52]
 [11 75]]


-- Fold 2 --
Train -> 1: 52.40% | 0: 47.60% (n=1000)
Validation -> 1: 57.33% | 0: 42.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 131ms/step
[BX][Fold 2] AUC: 0.6005 | F1: 0.6132 | MCC: -0.0753 | ACC: 0.4533
Confusion matrix:
[[ 3 77]
 [ 5 65]]


-- Fold 3 --
Train -> 1: 53.80% | 0: 46.20% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 131ms/step
[BX][Fold 3] AUC: 0.5621 | F1: 0.6082 | MCC: 0.1262 | ACC: 0.5533
Confusion matrix:
[[31 47]
 [20 52]]


-- Fold 4 --
Train -> 1: 54.10% | 0: 45.90% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1

...


-- Fold 1 --
Train -> 1: 50.90% | 0: 49.10% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 168ms/step
[VRTX][Fold 1] AUC: 0.4971 | F1: 0.6054 | MCC: 0.0458 | ACC: 0.5133
Confusion matrix:
[[21 56]
 [17 56]]


-- Fold 2 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 152ms/step
[VRTX][Fold 2] AUC: 0.4810 | F1: 0.6425 | MCC: 0.0000 | ACC: 0.4733
Confusion matrix:
[[ 0 79]
 [ 0 71]]


-- Fold 3 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step
[VRTX][Fold 3] AUC: 0.4262 | F1: 0.7048 | MCC: -0.0941 | ACC: 0.5533
Confusion matrix:
[[ 3 58]
 [ 9 80]]


-- Fold 4 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 59.33% | 0: 40.67% (n=150)
Test -> 1: 59.33% | 0: 40.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step
[WDC][Fold 1] AUC: 0.4753 | F1: 0.5525 | MCC: -0.0931 | ACC: 0.4600
Confusion matrix:
[[19 55]
 [26 50]]


-- Fold 2 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 141ms/step
[WDC][Fold 2] AUC: 0.4884 | F1: 0.6275 | MCC: 0.0263 | ACC: 0.4933
Confusion matrix:
[[10 68]
 [ 8 64]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step
[WDC][Fold 3] AUC: 0.5039 | F1: 0.6605 | MCC: 0.0809 | ACC: 0.5133
Confusion matrix:
[[ 6 70]
 [ 3 71]]


-- Fold 4 --
Train -> 1: 51.00% | 0: 49.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 50.00% | 0: 50.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-1870896826.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 43.33% | 0: 56.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step
[WFC][Fold 1] AUC: 0.5013 | F1: 0.5169 | MCC: -0.0926 | ACC: 0.4267
Confusion matrix:
[[18 67]
 [19 46]]


-- Fold 2 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 43.33% | 0: 56.67% (n=150)
Test -> 1: 52.67% | 0: 47.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step
[WFC][Fold 2] AUC: 0.4801 | F1: 0.2430 | MCC: -0.0599 | ACC: 0.4600
Confusion matrix:
[[56 15]
 [66 13]]


-- Fold 3 --
Train -> 1: 48.80% | 0: 51.20% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 141ms/step
[WFC][Fold 3] AUC: 0.4834 | F1: 0.6096 | MCC: -0.0071 | ACC: 0.5133
Confusion matrix:
[[20 49]
 [24 57]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Dominant Class Baseline Training
# ======================

def train_dominant_class_baseline(merged, config):
    set_seed(config.get('seed', 0))
    window_size = config['window_size']
    price_cols = config['price_cols']
    macro_cols = config['macro_cols']
    sent_cols = config['sent_cols']
    target = config['target']
    feature_cols = config['feature_cols'] or [f"{c}_logret" for c in price_cols] + [f"{c}_ret" for c in macro_cols]

    results = []

    for symbol in merged['Stock_symbol'].unique():
        print(f"\n==== Training for: {symbol} ====")
        df_symbol = merged[merged['Stock_symbol'] == symbol].copy()

        # Compute returns
        for col in price_cols:
            safe = df_symbol[col].replace(0, np.nan)
            df_symbol[f'{col}_logret'] = np.log(safe) - np.log(safe.shift(1))
        for col in macro_cols:
            df_symbol[f'{col}_ret'] = df_symbol[col].pct_change()
        df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')

        Xp_full, Xs_full, y_full = make_sequences_dual(df_symbol, window_size, feature_cols, sent_cols, target)
        if len(Xp_full) == 0:
            print(f"No sequences for {symbol}, skipping.")
            continue

        X_idx = np.arange(len(Xp_full))

        fold = 0
        for train_idx, val_idx, test_idx in overlapping_fixed_tscv(
                n_samples=len(X_idx),
                n_splits=config.get('n_splits', 5),
                train_size=config.get('train_size', 1000),
                val_size=config.get('val_size', 150),
                test_size=config.get('test_size', 150)):

            fold += 1
            print(f"\n-- Fold {fold} --")
            y_train, y_test = y_full[train_idx], y_full[test_idx]

            def dist(name, y):
                ones = np.sum(y == 1)
                zeros = np.sum(y == 0)
                print(f"{name} -> 1: {ones/len(y)*100:.2f}% | 0: {zeros/len(y)*100:.2f}% (n={len(y)}) ")

            dist("Train", y_train)
            dist("Test", y_test)

            # Determine dominant class in training data
            counts = np.bincount(y_test.astype(int))
            dominant_class = np.argmax(counts)

            # Baseline predictions: predict dominant class for all test samples
            y_pred = np.full_like(y_test, dominant_class)

            # For AUC, if dominant_class is 1, y_prob is all 1s. If 0, all 0s.
            # A constant prediction usually results in an undefined or 0.5 AUC, depending on the implementation.
            # For a strictly 'dominant class' prediction, y_prob should reflect that.
            y_prob = np.full_like(y_test, float(dominant_class))

            auc = roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) > 1 and len(np.unique(y_prob)) > 1 else 0.5
            f1 = f1_score(y_test, y_pred)
            mcc = matthews_corrcoef(y_test, y_pred)
            acc = np.mean(y_pred.flatten() == y_test.flatten())
            cm = confusion_matrix(y_test, y_pred)

            print(f"[{symbol}][Fold {fold}] BASELINE AUC: {auc:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | ACC: {acc:.4f}")
            print(f"Confusion matrix:\n{cm}\n")

            results.append({
                'Symbol': symbol,
                'Fold': fold,
                'Test_AUC': auc,
                'Test_F1': f1,
                'Test_MCC': mcc,
                'Test_ACC': acc
            })

    results_df = pd.DataFrame(results)
    return results_df

# ======================
# Baseline Config
# ======================
CONFIG_BASELINE = {
    "seed": 42,
    "window_size": 5,
    "price_cols": ['open', 'high', 'low', 'close', 'volume', 'DTWEXBGS', 'Price_x', 'WTI Crude Oil Price/Barrel'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "n_splits": 5,
    "train_size": 1000,
    "val_size": 150, # Not strictly used in baseline, but kept for consistency with tscv generator
    "test_size": 150,
}

# ======================
# Run Baseline
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    baseline_results_df = train_dominant_class_baseline(merged, CONFIG_BASELINE)

    # Summarize
    summarize_results(baseline_results_df)

    # Return or use results_df
    # baseline_results_df # now available for further processing


==== Training for: AAL ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-3276319099.py:26: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000) 
Test -> 1: 42.67% | 0: 57.33% (n=150) 
[AAL][Fold 1] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5733
Confusion matrix:
[[86  0]
 [64  0]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000) 
Test -> 1: 53.33% | 0: 46.67% (n=150) 
[AAL][Fold 2] BASELINE AUC: 0.5000 | F1: 0.6957 | MCC: 0.0000 | ACC: 0.5333
Confusion matrix:
[[ 0 70]
 [ 0 80]]


-- Fold 3 --
Train -> 1: 47.90% | 0: 52.10% (n=1000) 
Test -> 1: 44.00% | 0: 56.00% (n=150) 
[AAL][Fold 3] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5600
Confusion matrix:
[[84  0]
 [66  0]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000) 
Test -> 1: 45.33% | 0: 54.67% (n=150) 
[AAL][Fold 4] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5467
Confusion matrix:
[[82  0]
 [68  0]]


-- Fold 5 --
Train -> 1: 47.30% | 0: 52.70% (n=1000) 
Test -> 1: 49.33% | 0: 50.67% (n=150) 
[AAL][Fold 5] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.506

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-3276319099.py:26: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000) 
Test -> 1: 57.33% | 0: 42.67% (n=150) 
[BX][Fold 1] BASELINE AUC: 0.5000 | F1: 0.7288 | MCC: 0.0000 | ACC: 0.5733
Confusion matrix:
[[ 0 64]
 [ 0 86]]


-- Fold 2 --
Train -> 1: 52.40% | 0: 47.60% (n=1000) 
Test -> 1: 46.67% | 0: 53.33% (n=150) 
[BX][Fold 2] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5333
Confusion matrix:
[[80  0]
 [70  0]]


-- Fold 3 --
Train -> 1: 53.80% | 0: 46.20% (n=1000) 
Test -> 1: 48.00% | 0: 52.00% (n=150) 
[BX][Fold 3] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5200
Confusion matrix:
[[78  0]
 [72  0]]


-- Fold 4 --
Train -> 1: 54.10% | 0: 45.90% (n=1000) 
Test -> 1: 54.00% | 0: 46.00% (n=150) 
[BX][Fold 4] BASELINE AUC: 0.5000 | F1: 0.7013 | MCC: 0.0000 | ACC: 0.5400
Confusion matrix:
[[ 0 69]
 [ 0 81]]


==== Training for: CMCSA ====


...


-- Fold 1 --
Train -> 1: 50.90% | 0: 49.10% (n=1000) 
Test -> 1: 48.67% | 0: 51.33% (n=150) 
[VRTX][Fold 1] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5133
Confusion matrix:
[[77  0]
 [73  0]]


-- Fold 2 --
Train -> 1: 51.60% | 0: 48.40% (n=1000) 
Test -> 1: 47.33% | 0: 52.67% (n=150) 
[VRTX][Fold 2] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5267
Confusion matrix:
[[79  0]
 [71  0]]


-- Fold 3 --
Train -> 1: 51.60% | 0: 48.40% (n=1000) 
Test -> 1: 59.33% | 0: 40.67% (n=150) 
[VRTX][Fold 3] BASELINE AUC: 0.5000 | F1: 0.7448 | MCC: 0.0000 | ACC: 0.5933
Confusion matrix:
[[ 0 61]
 [ 0 89]]


-- Fold 4 --
Train -> 1: 50.00% | 0: 50.00% (n=1000) 
Test -> 1: 59.33% | 0: 40.67% (n=150) 
[VRTX][Fold 4] BASELINE AUC: 0.5000 | F1: 0.7448 | MCC: 0.0000 | ACC: 0.5933
Confusion matrix:
[[ 0 61]
 [ 0 89]]


==== Training for: WDC ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-3276319099.py:26: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 51.30% | 0: 48.70% (n=1000) 
Test -> 1: 50.67% | 0: 49.33% (n=150) 
[WDC][Fold 1] BASELINE AUC: 0.5000 | F1: 0.6726 | MCC: 0.0000 | ACC: 0.5067
Confusion matrix:
[[ 0 74]
 [ 0 76]]


-- Fold 2 --
Train -> 1: 51.80% | 0: 48.20% (n=1000) 
Test -> 1: 48.00% | 0: 52.00% (n=150) 
[WDC][Fold 2] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5200
Confusion matrix:
[[78  0]
 [72  0]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000) 
Test -> 1: 49.33% | 0: 50.67% (n=150) 
[WDC][Fold 3] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5067
Confusion matrix:
[[76  0]
 [74  0]]


-- Fold 4 --
Train -> 1: 51.00% | 0: 49.00% (n=1000) 
Test -> 1: 50.00% | 0: 50.00% (n=150) 
[WDC][Fold 4] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5000
Confusion matrix:
[[75  0]
 [75  0]]


==== Training for: WFC ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-3276319099.py:26: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.40% | 0: 50.60% (n=1000) 
Test -> 1: 43.33% | 0: 56.67% (n=150) 
[WFC][Fold 1] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5667
Confusion matrix:
[[85  0]
 [65  0]]


-- Fold 2 --
Train -> 1: 49.30% | 0: 50.70% (n=1000) 
Test -> 1: 52.67% | 0: 47.33% (n=150) 
[WFC][Fold 2] BASELINE AUC: 0.5000 | F1: 0.6900 | MCC: 0.0000 | ACC: 0.5267
Confusion matrix:
[[ 0 71]
 [ 0 79]]


-- Fold 3 --
Train -> 1: 48.80% | 0: 51.20% (n=1000) 
Test -> 1: 54.00% | 0: 46.00% (n=150) 
[WFC][Fold 3] BASELINE AUC: 0.5000 | F1: 0.7013 | MCC: 0.0000 | ACC: 0.5400
Confusion matrix:
[[ 0 69]
 [ 0 81]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000) 
Test -> 1: 47.33% | 0: 52.67% (n=150) 
[WFC][Fold 4] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5267
Confusion matrix:
[[79  0]
 [71  0]]


-- Fold 5 --
Train -> 1: 50.10% | 0: 49.90% (n=1000) 
Test -> 1: 48.67% | 0: 51.33% (n=150) 
[WFC][Fold 5] BASELINE AUC: 0.5000 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.513